In [19]:
import numpy as np
import cv2

# çıktı dosyalarında sıra ile değerler:
# frame, bounding_boxes(vec), feature_locations(x,y), features(vec), label


# box = xmin, xmax, ymin, ymax
def is_inside(key_point, box):
    x = key_point[0]
    y = key_point[1]

    return (x > box[0] and x < box[1] ) and (y > box[2] and y < box[3] )

# resimdeki feature için [nokta, feature, label] döner
def extract_features(image, bounding_boxes):
    sift = cv2.SIFT_create()
    key_points, descriptors = sift.detectAndCompute(image, None)
    assert len(key_points) > 0
    points = []
    feats = []
    labels = []
    for i, feat in enumerate(key_points):
        box_count = 0 #içinde olduğu box sayısı
        for box in bounding_boxes:
            if(is_inside(feat.pt, box)):
                box_count = box_count + 1
        if box_count >= 1: # araba var
            points.append(feat.pt)
            feats.append(descriptors[i])
            labels.append(1)
        elif box_count == 0: # yok
            points.append(feat.pt)
            feats.append(descriptors[i])
            labels.append(0)
        # else drop 
                
    return points, feats, labels


train_set = np.load("train_frames.npy", allow_pickle=True)
points = []
features = []
labels = []
for i in range(len(train_set)):
    row = train_set[i]
    image = cv2.imread("images/"+row[0])
    point, feat, label = extract_features(image, row[1])

    points.append(point)
    features.append(feat)
    labels.append(label)

points = np.array(points, dtype=object).reshape(-1,1)
features = np.array(features, dtype=object).reshape(-1,1)
labels = np.array(labels, dtype=object).reshape(-1,1)



train_set = np.hstack((train_set, points, features, labels))
# train_set[:,1] = [x[0] for x in train_set[:,1]] # sadece en büyük box kalsın

x_col = train_set[:,3]
y_col = train_set[:,4]
x = []
y = []

for i in range(len(x_col)):
    for j in range(len(x_col[i])):
        x.append(x_col[i][j])
        y.append(y_col[i][j])

X = np.array(x)
Y = np.array(y)
np.save("X.npy",X)
np.save("Y.npy",Y)


#---------------------------------------------------------

train_set = np.load("val_frames.npy", allow_pickle=True)
points = []
features = []
labels = []
for i in range(len(train_set)):
    row = train_set[i]
    image = cv2.imread("images/"+row[0])
    point, feat, label = extract_features(image, row[1])

    points.append(point)
    features.append(feat)
    labels.append(label)

points = np.array(points, dtype=object).reshape(-1,1)
features = np.array(features, dtype=object).reshape(-1,1)
labels = np.array(labels, dtype=object).reshape(-1,1)

train_set = np.hstack((train_set, points, features, labels))
# train_set[:,1] = [x[0] for x in train_set[:,1]] # sadece en büyük box kalsın

np.save("val_frames_with_features", train_set)


#---------------------------------------------------------


train_set = np.load("test_frames.npy", allow_pickle=True)
points = []
features = []
labels = []
for i in range(len(train_set)):
    row = train_set[i]
    image = cv2.imread("images/"+row[0])
    point, feat, label = extract_features(image, row[1])

    points.append(point)
    features.append(feat)
    labels.append(label)

points = np.array(points, dtype=object).reshape(-1,1)
features = np.array(features, dtype=object).reshape(-1,1)
labels = np.array(labels, dtype=object).reshape(-1,1)

train_set = np.hstack((train_set, points, features, labels))
# train_set[:,1] = [x[0] for x in train_set[:,1]] # sadece en büyük box kalsın

np.save("test_frames_with_features", train_set)




In [23]:
# Windowlar için feature çıkar. Sıra ile 128 - 64 - 32 - 16
import cv2
import numpy as np

# xmin, xmax, ymin, ymax şeklinde olmalı kutular
def calculate_iou(box1, box2, threshold = 0.1):
    x1_min, x1_max, y1_min, y1_max = box1
    x2_min, x2_max, y2_min, y2_max = box2

    xmin = max(x1_min, x2_min)
    xmax = min(x1_max, x2_max)
    ymin = max(y1_min, y2_min)
    ymax = min(y1_max, y2_max)

    width = max(0, xmax- xmin)
    height = max(0, ymax- ymin)

    inter_area = width * height
    
    box1_area = (x1_max - x1_min) * (y1_max - y1_min)
    box1_area = max(0,box1_area)

    box2_area = (x2_max - x2_min) * (y2_max - y2_min)
    box2_area = max(0,box2_area)

    iou = inter_area / (box1_area + box2_area - inter_area)
    if iou <= 0:
        return False
    return iou >= threshold



def extract_window_features(image, window_size):
    sift = cv2.SIFT_create(nfeatures = ((window_size//16)**2) * 4)  
    win_points = []
    win_features = []
    x = 128//window_size
    x = x + (x-1)
    y = x
    for j in range(y):
        points = []
        features = []
        for k in range(x):

            edge_length = window_size // 2
            x_edge_start = edge_length * k
            y_edge_start = edge_length * j
            window = image[ y_edge_start : y_edge_start + window_size, x_edge_start : x_edge_start + window_size]
            kp, dc = sift.detectAndCompute(window, None)
            for index, feat in enumerate(kp):
                points.append(feat.pt)
                features.append(dc[index])


            win_points.append(points)
            win_features.append(features)

    return win_points, win_features

def extract_all_window_features(image, boxes):
    win_sizes = [128, 64, 32, 16]

    win_features = []

    for i, win_size in enumerate(win_sizes):
        points, features = extract_window_features(image, win_size)

        # Window labels
        x = 128//win_size
        x = x + (x-1)
        y = x
        labels = []
        for j in range(y):
            for k in range(x):
                found = False
                for box in boxes:
                    edge_length = win_size // 2
                    x_min = edge_length * k
                    x_max = x_min + win_size
                    
                    y_min = edge_length * j
                    y_max = y_min + win_size

                    if calculate_iou(box, (x_min, x_max, y_min, y_max)):
                        found = True
                        break
                if found:
                    labels.append(1)
                else:
                    labels.append(0)



        win_features.append({
            "win_size": win_size,
            "points": points,
            "features": features,
            "labels": labels
        })
    
    return win_features




train_set = np.load("val_frames.npy", allow_pickle=True)
win_features = []
for i in range(len(train_set)):
    image = cv2.imread("images/"+ train_set[i][0])
    feats = extract_all_window_features(image, train_set[i][1])
    win_features.append(feats)

win_features = np.array(win_features, dtype=object)

train_set = np.hstack((train_set, win_features))

np.save("val_frames_with_window_features", train_set)


train_set = np.load("test_frames.npy", allow_pickle=True)
win_features = []
for i in range(len(train_set)):
    image = cv2.imread("images/"+ train_set[i][0])
    feats = extract_all_window_features(image, train_set[i][1])
    win_features.append(feats)

win_features = np.array(win_features, dtype=object)

train_set = np.hstack((train_set, win_features))

np.save("test_frames_with_window_features", train_set)

